In [2]:
"""
RUTA ÓPTIMA APP - SISTEMA LOGÍSTICO
Este programa simula un sistema logístico para la gestión de clientes, paquetes y envíos utilizando Python.
Se implementa Programación Orientada a Objetos (POO) con clases, herencia y encapsulamiento, junto con una base de datos SQLite para almacenar la información.
El sistema permite realizar operaciones CRUD (crear, leer, actualizar y eliminar datos), además de consultas JOIN para relacionar múltiples tablas.
"""

# ======================
# IMPORTACIÓN DE LIBRERÍA
# ======================

# Importamos sqlite3 para poder crear y manejar bases de datos en Python
import sqlite3


# ======================
# FUNCIÓN DE CONEXIÓN
# ======================

# Esta función crea (si no existe) y conecta con la base de datos
def conectar():
    # "ruta_optima.db" es el archivo donde se guardarán los datos
    return sqlite3.connect("ruta_optima.db")


# ======================
# CLASES PADRE (POO)
# ======================

# Clase Cliente: representa a cada persona que usa el sistema
class Cliente:
    def __init__(self, nombre):
        # Atributo privado (__nombre) → encapsulamiento
        # Solo se puede acceder mediante métodos get/set
        self.__nombre = nombre

    # Método GET → permite leer el nombre
    def get_nombre(self):
        return self.__nombre

    # Método SET → permite modificar el nombre
    def set_nombre(self, nombre):
        self.__nombre = nombre


# Clase Paquete: representa un paquete logístico
class Paquete:
    def __init__(self, peso, tipo):
        # Atributos privados
        self.__peso = peso   # peso del paquete
        self.__tipo = tipo   # tipo: Ligero o Pesado

    # Método GET → obtener peso
    def get_peso(self):
        return self.__peso

    # Método GET → obtener tipo
    def get_tipo(self):
        return self.__tipo

    # Método propio → calcula costo del envío
    def calcular_costo(self):
        # Lógica: costo = peso * 1000
        return self.__peso * 1000


# Clase Envio: representa el envío asociado a un paquete
class Envio:
    def __init__(self, paquete_id):
        # Guarda el ID del paquete relacionado
        # Esto conecta con la base de datos
        self.paquete_id = paquete_id


# ======================
# CLASES HIJO (HERENCIA)
# ======================

# ClientePremium hereda de Cliente
class ClientePremium(Cliente):
    # Método adicional exclusivo de clientes premium
    def aplicar_descuento(self):
        return 0.1  # 10% de descuento


# PaqueteLigero hereda de Paquete
class PaqueteLigero(Paquete):
    pass  # no agrega nada nuevo, hereda todo


# PaquetePesado hereda de Paquete
class PaquetePesado(Paquete):
    pass


# EnvioExpress hereda de Envio
class EnvioExpress(Envio):
    pass


# ======================
# CREACIÓN DE TABLAS
# ======================

def crear_tablas():
    try:
        # Conectamos a la base de datos
        conn = conectar()
        cursor = conn.cursor()

        # ----------------------
        # TABLA CLIENTES
        # ----------------------
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS clientes (
            id_cliente INTEGER PRIMARY KEY AUTOINCREMENT, -- identificador único
            nombre TEXT                                   -- nombre del cliente
        )
        """)

        # ----------------------
        # TABLA PAQUETES
        # ----------------------
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS paquetes (
            id_paquete INTEGER PRIMARY KEY AUTOINCREMENT, -- identificador único
            peso REAL,                                   -- peso del paquete
            tipo TEXT,                                   -- Ligero o Pesado
            id_cliente INTEGER,                          -- FK (relación con cliente)
            FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
        )
        """)

        # ----------------------
        # TABLA ENVIOS
        # ----------------------
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS envios (
            id_envio INTEGER PRIMARY KEY AUTOINCREMENT, -- identificador único
            id_paquete INTEGER,                        -- FK (relación con paquete)
            FOREIGN KEY(id_paquete) REFERENCES paquetes(id_paquete)
        )
        """)

        # Guardamos los cambios en la base de datos
        conn.commit()

        # Cerramos la conexión
        conn.close()

    except Exception as e:
        # Captura errores si algo falla
        print("Error creando tablas:", e)


# ======================
# INSERTAR DATOS INICIALES
# ======================

def insertar_datos():
    try:
        conn = conectar()
        cursor = conn.cursor()

        # Lista de clientes (cada tupla es un registro)
        clientes = [
            ("Ana",),
            ("Luis",),
            ("Carlos",),
            ("Sofia",),
            ("Marta",)
        ]

        # Lista de paquetes: (peso, tipo, id_cliente)
        paquetes = [
            (2.5, "Ligero", 1),
            (10.0, "Pesado", 2),
            (1.2, "Ligero", 3),
            (8.0, "Pesado", 4),
            (3.5, "Ligero", 5)
        ]

        # Insertamos varios clientes a la vez
        cursor.executemany(
            "INSERT INTO clientes(nombre) VALUES (?)",
            clientes
        )

        # Insertamos paquetes relacionados con clientes
        cursor.executemany(
            "INSERT INTO paquetes(peso, tipo, id_cliente) VALUES (?, ?, ?)",
            paquetes
        )

        conn.commit()
        conn.close()

    except Exception as e:
        print("Error insertando datos:", e)


# Insertar envíos
def insertar_envios():
    try:
        conn = conectar()
        cursor = conn.cursor()

        # Cada envío está ligado a un paquete
        envios = [(1,), (2,), (3,), (4,), (5,)]

        cursor.executemany(
            "INSERT INTO envios(id_paquete) VALUES (?)",
            envios
        )

        conn.commit()
        conn.close()

    except Exception as e:
        print("Error insertando envíos:", e)


# ======================
# CRUD CLIENTES
# ======================

# CREATE → Crear cliente
def crear_cliente(nombre):
    try:
        # Creamos objeto usando POO
        cliente = Cliente(nombre)

        conn = conectar()
        cursor = conn.cursor()

        # Insertamos en la base de datos
        cursor.execute(
            "INSERT INTO clientes(nombre) VALUES (?)",
            (cliente.get_nombre(),)  # usamos método GET
        )

        conn.commit()
        conn.close()

    except Exception as e:
        print("Error creando cliente:", e)


# READ → Ver clientes
def ver_clientes():
    try:
        conn = conectar()
        cursor = conn.cursor()

        # Consulta para traer todos los clientes
        cursor.execute("SELECT * FROM clientes")

        print("\nCLIENTES:")
        for fila in cursor.fetchall():
            print(fila)

        conn.close()

    except Exception as e:
        print("Error leyendo clientes:", e)


# UPDATE → Actualizar cliente
def actualizar_cliente(id_cliente, nombre):
    try:
        conn = conectar()
        cursor = conn.cursor()

        # Modificamos el nombre según el ID
        cursor.execute(
            "UPDATE clientes SET nombre=? WHERE id_cliente=?",
            (nombre, id_cliente)
        )

        conn.commit()
        conn.close()

    except Exception as e:
        print("Error actualizando cliente:", e)


# DELETE → Eliminar cliente
def eliminar_cliente(id_cliente):
    try:
        conn = conectar()
        cursor = conn.cursor()

        # Eliminamos cliente por ID
        cursor.execute(
            "DELETE FROM clientes WHERE id_cliente=?",
            (id_cliente,)
        )

        conn.commit()
        conn.close()

    except Exception as e:
        print("Error eliminando cliente:", e)


# ======================
# CONSULTA JOIN
# ======================

def ver_envios():
    try:
        conn = conectar()
        cursor = conn.cursor()

        # JOIN entre 3 tablas:
        # envios → paquetes → clientes
        cursor.execute("""
        SELECT clientes.nombre, paquetes.peso, paquetes.tipo
        FROM envios
        JOIN paquetes ON envios.id_paquete = paquetes.id_paquete
        JOIN clientes ON paquetes.id_cliente = clientes.id_cliente
        """)

        print("\nENVÍOS:")
        for fila in cursor.fetchall():
            print("Cliente:", fila[0], "| Peso:", fila[1], "| Tipo:", fila[2])

        conn.close()

    except Exception as e:
        print("Error en JOIN:", e)


# ======================
# MENÚ INTERACTIVO
# ======================

def menu():
    # Inicializamos sistema (crea tablas y datos)
    crear_tablas()
    insertar_datos()
    insertar_envios()

    # Bucle infinito para interacción
    while True:
        print("\n--- MENÚ ---")
        print("1. Ver clientes")
        print("2. Agregar cliente")
        print("3. Actualizar cliente")
        print("4. Eliminar cliente")
        print("5. Ver envíos (JOIN)")
        print("6. Salir")

        try:
            # Usuario elige opción
            opcion = int(input("Elige una opción: "))

            # Ejecuta según la opción
            if opcion == 1:
                ver_clientes()

            elif opcion == 2:
                nombre = input("Nombre: ")
                crear_cliente(nombre)

            elif opcion == 3:
                id_cliente = int(input("ID: "))
                nombre = input("Nuevo nombre: ")
                actualizar_cliente(id_cliente, nombre)

            elif opcion == 4:
                id_cliente = int(input("ID: "))
                eliminar_cliente(id_cliente)

            elif opcion == 5:
                ver_envios()

            elif opcion == 6:
                print("Saliendo...")
                break

        except Exception as e:
            print("Error:", e)


# ======================
# EJECUCIÓN
# ======================

# Inicia todo el sistema
menu()


--- MENÚ ---
1. Ver clientes
2. Agregar cliente
3. Actualizar cliente
4. Eliminar cliente
5. Ver envíos (JOIN)
6. Salir

--- MENÚ ---
1. Ver clientes
2. Agregar cliente
3. Actualizar cliente
4. Eliminar cliente
5. Ver envíos (JOIN)
6. Salir

ENVÍOS:
Cliente: Ana | Peso: 2.5 | Tipo: Ligero
Cliente: Luis | Peso: 10.0 | Tipo: Pesado
Cliente: Carlos | Peso: 1.2 | Tipo: Ligero
Cliente: Sofia | Peso: 8.0 | Tipo: Pesado
Cliente: Marta | Peso: 3.5 | Tipo: Ligero
Cliente: Ana | Peso: 2.5 | Tipo: Ligero
Cliente: Luis | Peso: 10.0 | Tipo: Pesado
Cliente: Carlos | Peso: 1.2 | Tipo: Ligero
Cliente: Sofia | Peso: 8.0 | Tipo: Pesado
Cliente: Marta | Peso: 3.5 | Tipo: Ligero

--- MENÚ ---
1. Ver clientes
2. Agregar cliente
3. Actualizar cliente
4. Eliminar cliente
5. Ver envíos (JOIN)
6. Salir
Saliendo...
